In [1]:
import os
import torch

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from torch.nn import Linear
from model.layers import Add, Clone
from model.ViT import Attention
import torch.nn.functional as F
from einops import rearrange
import torch.nn as nn
import torchvision.models as models
from torch_geometric.nn import GATv2Conv, LayerNorm
from model.ViT import Mlp, VisionTransformer


class GraphNN(nn.Module):
    def __init__(self, cell_dim=80, vit_depth=3, proto=False, ensemble=False):
        super(GraphNN, self).__init__()
        self.resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.resnet18 = torch.nn.Sequential(*list(self.resnet18.children())[:-1])
        
        self.embed_dim = 32 * 8
        self.head = 8
        self.dropout = 0.3
        
        self.conv1 = GATv2Conv(in_channels=512, out_channels=int(self.embed_dim/self.head), heads=self.head)
        self.norm1 = LayerNorm(in_channels=self.embed_dim)
        
        self.cell_transformer = VisionTransformer(num_classes=cell_dim, embed_dim=self.embed_dim, depth=vit_depth,
                                                  mlp_head=True, drop_rate=self.dropout, attn_drop_rate=self.dropout)
        self.proto = proto
        if self.proto:
            self.cell_proto = nn.Parameter(torch.zeros(1, 150, self.embed_dim))
            self.cell_qkv = Linear(self.embed_dim, self.embed_dim*2)
            self.cell_att = Attention(dim=self.embed_dim, num_heads=self.head, attn_drop=self.dropout, proj_drop=self.dropout)
            self.add2 = Add()
            self.clone2 = Clone()
            self.task_norm_3 = LayerNorm(self.embed_dim, eps=1e-6)
            self.task_norm_4 = LayerNorm(self.embed_dim, eps=1e-6)
            self.cell_att_W = Linear(self.embed_dim, self.embed_dim)
            torch.nn.init.xavier_uniform_(self.cell_proto)
            
        self.ensemble = ensemble
        if self.ensemble:
            self.spot_fc = Linear(in_features=512, out_features=256)
            self.spot_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
            self.local_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
            self.fused_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
        
    def forward(self, x, edge_index):
        x_spot = self.resnet18(x)
        x_spot = x_spot.squeeze()
        
        x_local = self.conv1(x=x_spot, edge_index=edge_index)
        x_local = self.norm1(x_local)
        
        x_local = x_local.unsqueeze(0)
        
        if self.proto:
            x_cell1, x_cell2 = self.clone2(x_local, 2)
            x_cell_qkv = self.cell_qkv(self.cell_proto)
            x_cell_k, x_cell_v = rearrange(x_cell_qkv, 'b n (qkv h d) -> qkv b h n d', qkv=2, h=8)
            x_cell = self.add2([x_cell1, self.cell_att_W(self.cell_att(x=x_cell2, out_k=x_cell_k, out_v=x_cell_v))])
            x_cell = self.task_norm_4(x_cell)
            x_cell = self.task_norm_3(x_cell + F.relu(x_cell))
        else:
            x_cell = x_local
        
        if self.ensemble:
            x_spot = self.spot_fc(x_spot)
            cell_predication_spot = self.spot_head(x_spot)
            x_local = x_local.squeeze(0)
            cell_prediction_local = self.local_head(x_local)
            cell_prediction_global, x_global = self.cell_transformer(x_cell)
            cell_prediction_global = cell_prediction_global.squeeze()
            x_global = x_global.squeeze()
            cell_prediction_fused = self.fused_head((x_spot+x_local+x_global)/3.0)
            # cell_prediction = (cell_predication_spot + cell_prediction_local + cell_prediction_global + cell_prediction_fused) / 4.0
            cell_prediction = (cell_predication_spot + cell_prediction_local*3.0 + cell_prediction_global + cell_prediction_fused) / 6.0
        else:  
            cell_prediction, _ = self.cell_transformer(x_cell)
            cell_prediction = cell_prediction.squeeze()
        
        # cell_prediction = torch.relu(cell_prediction)
        
        return cell_prediction

In [3]:
from glob import glob
tif_list = glob('/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/stnet/test*')
tif_list.sort()
test_slides = list()
for tif in tif_list:
    tif_path = tif.split('_')[-1].split('.')[0]
    test_slides.append(tif_path)
test_slides

['23209',
 '23268',
 '23269',
 '23270',
 '23272',
 '23277',
 '23287',
 '23288',
 '23377',
 '23450',
 '23506',
 '23508',
 '23567',
 '23803',
 '23810',
 '23895',
 '23901',
 '23903',
 '23944',
 '24044',
 '24105',
 '24220',
 '24223']

In [4]:
from torch_geometric.data import Batch
from torch_geometric.loader import NeighborLoader
import torch_geometric
torch_geometric.typing.WITH_PYG_LIB = False
from tqdm import tqdm
import numpy as np
from scipy.stats import pearsonr
import joblib
    
ensemble = True
vit_depth = 1
celltype_num = 39
model = GraphNN(cell_dim=celltype_num, vit_depth=1, ensemble=ensemble)
checkpoint = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/training/model_weights/breast_cross_source_epoch100_lr1e-4_2hop_ensemble_Trans1layer_GNNoutput50_onlycell_best_cell_all_abundance_average.pth")
model.load_state_dict(checkpoint)
model = model.to(device)    

for case in test_slides:
    print(case)
    data_path = "/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet"
    hop = 2
    subgraph_bs = 16   
    
    criterion = nn.MSELoss().to(device)
    
    test_set = "/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/stnet/test_leave_"+case+".txt"
    test_slides = open(test_set).read().split('\n')

    test_graph_list = list()
    for item in test_slides:
        test_graph_list.append(torch.load(os.path.join(data_path, item+'.pt')))
    test_dataset = Batch.from_data_list(test_graph_list)
    
    print(test_slides)
    # continue    

    test_loader = NeighborLoader(
        test_dataset,
        num_neighbors=[-1]*hop,
        batch_size=subgraph_bs,
        directed=False,
        input_nodes=None,
        shuffle=False,
        # num_workers=8,
        # pin_memory=True, 
        # prefetch_factor=2,
    )

    with torch.no_grad():
        model.eval()

        test_sample_num = 0
        test_cell_pred_array = []
        test_cell_label_array = []
        test_cell_pos_array = []
        test_loss, test_cell_abundance_loss = 0, 0
        
        for graph in tqdm(test_loader):
            x = graph.x.to(device)
            y = graph.y.to(device)
            pos = graph.pos.to(device)
            edge_index = graph.edge_index.to(device)
            cell_label = y[:, 250:]
            cell_pred = model(x=x, edge_index=edge_index)
            cell_loss = criterion(cell_pred, cell_label)

            loss = cell_loss
                
            center_num = len(graph.input_id)
            center_cell_label = cell_label[:center_num, :]
            center_cell_pred = cell_pred[:center_num, :]
            center_cell_pos = pos[:center_num, :]
            
            test_cell_label_array.append(center_cell_label.squeeze().cpu().detach().numpy())
            test_cell_pred_array.append(center_cell_pred.squeeze().cpu().detach().numpy())
            test_cell_pos_array.append(center_cell_pos.squeeze().cpu().detach().numpy())
            test_sample_num = test_sample_num + center_num
            
            test_loss += loss.item() * center_num
            test_cell_abundance_loss += cell_loss.item() * center_num

        test_cell_abundance_loss = test_cell_abundance_loss / test_sample_num
    
        if len(test_cell_pred_array[-1].shape) == 1:
            test_cell_pred_array[-1] = np.expand_dims(test_cell_pred_array[-1], axis=0)
        test_cell_pred_array = np.concatenate(test_cell_pred_array)
        if len(test_cell_label_array[-1].shape) == 1:
            test_cell_label_array[-1] = np.expand_dims(test_cell_label_array[-1], axis=0)
        test_cell_label_array = np.concatenate(test_cell_label_array)
        if len(test_cell_pos_array[-1].shape) == 1:
            test_cell_pos_array[-1] = np.expand_dims(test_cell_pos_array[-1], axis=0)
        test_cell_pos_array = np.concatenate(test_cell_pos_array)
    
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array[:, i], test_cell_label_array[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num

        print(case)
        print("saving " + "best cell all abundance average " + str(test_cell_abundance_all_pearson_average))
        
    
    Predictions = dict()

    for slide_no in range(len(test_slides)):
        indices = np.where(test_dataset.batch.numpy() == slide_no)
        test_cell_pred_array_sub = test_cell_pred_array[indices]
        test_cell_label_array_sub = test_cell_label_array[indices]
        test_cell_pos_arraay_sub = test_cell_pos_array[indices]
        
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array_sub[:, i], test_cell_label_array_sub[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num
        
        Predictions[test_slides[slide_no]] = {
            'cell_abundance_predictions': test_cell_pred_array_sub,
            'cell_abundance_labels': test_cell_label_array_sub,
            'coords': test_cell_pos_arraay_sub,
        }
        
        print(test_slides[slide_no]) 
        print(test_cell_abundance_all_pearson_average)    

    save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/breast_cross_source/breast_cross_source_epoch100_lr1e-4_2hop_ensemble_Trans1layer_GNNoutput50_onlycell_"+case+"_best_cell_all_abundance_average.pkl"
    joblib.dump(Predictions, save_path)

23209


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


['23209_C1', '23209_C2', '23209_D1']


100%|██████████| 59/59 [00:06<00:00,  8.50it/s]


23209
saving best cell all abundance average 0.2096542760811238
23209_C1
0.1603187837486133
23209_C2
0.21275385244612552
23209_D1
0.2662713402849251
23268
['23268_C1', '23268_C2', '23268_D1']


...

24105
saving best cell all abundance average 0.1426971198588893
24105_C1
0.21138833946348476
24105_C2
0.1787279477524114
24105_D1
0.054484229593141704
24220
['24220_D2', '24220_E1', '24220_E2']


100%|██████████| 82/82 [00:08<00:00, 10.03it/s]


24220
saving best cell all abundance average 0.2548814402956265
24220_D2
0.26931298742594606
24220_E1
0.34046575645100136
24220_E2
0.14243009746311824
24223
['24223_D2', '24223_E1', '24223_E2']


100%|██████████| 69/69 [00:06<00:00, 10.89it/s]

24223
saving best cell all abundance average 0.2678348323899928
24223_D2
0.23428994454229887
24223_E1
0.3102746994016344
24223_E2
0.2636717974015863
